<a href="https://colab.research.google.com/github/yoshiagent/automaticControl-colab/blob/main/notebooks/AnalyticSolution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 微分方程式的解析解 Analytical Solution of ODEs

對照網頁：[自動控制｜解析解](https://yoshiagent.github.io/automaticControl/AS.html)

原課程用 MATLAB 的 Symbolic Math Toolbox（`dsolve`）求常係數線性常微分方程式的解，再用 `plot` 畫出暫態解、穩態解與全解。
這本筆記本先列出原本的 MATLAB 程式，再給逐段對應的 Python 版本：
用 **sympy** 的 `dsolve` 重新求解（對應 MATLAB `dsolve`），再用 **scipy** 的數值積分 `solve_ivp` 交叉驗證。

| 原檔 | 方程式 | 本筆記本 |
|---|---|---|
| `sys2ndfree.m` | $\ddot y+2\zeta\omega_n\dot y+\omega_n^2y=0$（標準二階系統自由響應） | 第 1 節 |
| `Osci1.m` | $\ddot x+4x=0,\ x(0)=1,\ \dot x(0)=6$ | 第 2 節 |
| `nonOsc.m` | $\ddot x+5\dot x+6x=0,\ x(0)=1,\ \dot x(0)=0$ | 第 2 節 |
| `TranSteady.m` | $\ddot x+5\dot x+6x=10\sin t$（暫態解＋穩態解） | 第 3 節 |
| `Unstable.m` | $\ddot x-5\dot x+6x=10\sin t$（不穩定） | 第 4 節 |
| `Resonance.m` | $\ddot x+4x=8\sin 2t$（共振） | 第 5 節 |
| `noResonance.m` | $\ddot x+4x=3\sin t$（不共振） | 第 5 節 |
| `Osci.m` | $\ddot x+4x=4$（步階輸入、無阻尼） | 第 5 節 |

只用 Colab 內建的 numpy、scipy、sympy、plotly。圖上的文字用英文。

In [1]:
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
import plotly.graph_objects as go

t = sp.symbols("t", real=True)
x = sp.Function("x")


def show(fig, title, xlab="t (s)", ylab="x(t)", h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()


def dsolve(a1, a0, f, x0=None, v0=None):
    """仿 MATLAB dsolve：解 x'' + a1 x' + a0 x = f(t)；有給初始條件就代入（沒給就留下積分常數 C1、C2）"""
    ode = sp.Eq(x(t).diff(t, 2) + a1 * x(t).diff(t) + a0 * x(t), f)
    ics = None if x0 is None else {x(0): x0, x(t).diff(t).subs(t, 0): v0}
    return sp.dsolve(ode, x(t), ics=ics).rhs


def num(expr, tt):
    """把 sympy 式子在 numpy 時間軸上取值（常數也展開成陣列）"""
    return np.broadcast_to(sp.lambdify(t, expr, "numpy")(tt), tt.shape).astype(float)


def check(label, a1, a0, f, x0, v0, x_matlab, T):
    """比對三者：原 MATLAB 程式裡寫的解、sympy dsolve 的解、數值積分 solve_ivp"""
    sol = dsolve(a1, a0, f, x0, v0)
    same = sp.simplify(sol - x_matlab) == 0
    fn = sp.lambdify(t, f, "numpy")
    tt = np.linspace(0, T, 401)
    ode = lambda ti, y: [y[1], float(fn(ti)) - a1 * y[1] - a0 * y[0]]
    yn = solve_ivp(ode, (0, T), [x0, v0], t_eval=tt, rtol=1e-11, atol=1e-12).y[0]
    err = np.max(np.abs(yn - num(x_matlab, tt)))
    print(f"{label}: sympy dsolve -> x(t) = {sol}")
    print(f"{label}: agree with sympy = {same}, max |x - solve_ivp| = {err:.1e}")
    return sol


def plot3(tt, parts, title):
    """仿原程式 plot(t,xt,'b',t,xs,'r',t,x,'k')：藍、紅、黑三條線"""
    fig = go.Figure()
    for (name, y), c in zip(parts, ["#1f77b4", "#d62728", "black"]):
        fig.add_scatter(x=tt, y=y, name=name, line=dict(color=c, width=2.5 if c == "black" else 1.8))
    show(fig, title)

## 1. 標準二階系統的自由響應 `sys2ndfree.m`

```matlab
% Created by T.Y.Hwang, M.E. Chinese Culture University,
% 2003.03.08
z=0.3;  % damping ratio
wn=2;
wd=wn*sqrt(1-z^2);
y0=1;
v0=0;
dt=0.001;
t=0:dt:10;
rt=roots([1 2*z*wn wn^2])
if (z<1) & (z>-1)
  c1=y0;
  c2=  (v0+z*wn*y0)/wd;
  y0=sqrt(c1^2+c2^2)*exp(-z*wn*t);
  y1=-sqrt(c1^2+c2^2)*exp(-z*wn*t);
  y=exp(-z*wn*t).*(c2*sin(wd*t)+c1*cos(wd*t));
  plot(t,y,t,y0,':r',t,y1,':r')
elseif (z==1) | (z==-1)
  c1=y0;
  c2=v0+wn*z*y0;
  y=(c1+c2*t).*exp(-wn*z*t);
  plot(t,y)
elseif (z>1) | (z<-1)
    z1=sqrt(z^2-1);
    c1=(v0+(z+z1)*wn*y0)/(2*wn*z1);
    c2=(-v0-(z-z1)*wn*y0)/(2*wn*z1);
    y=exp(-z*wn*t).*(c1*exp(z1*wn*t)+c2*exp(-z1*wn*t));
    plot(t,y)
end
grid
```

方程式 $\ddot y+2\zeta\omega_n\dot y+\omega_n^2y=0$，初始條件 $y(0)=y_0,\ \dot y(0)=v_0$。依阻尼比 $\zeta$ 分三種情形：

* 欠阻尼 $|\zeta|<1$：$y=e^{-\zeta\omega_nt}\left(c_1\cos\omega_dt+c_2\sin\omega_dt\right)$，$\omega_d=\omega_n\sqrt{1-\zeta^2}$，
  $c_1=y_0$、$c_2=\dfrac{v_0+\zeta\omega_ny_0}{\omega_d}$；包絡線 $\pm\sqrt{c_1^2+c_2^2}\,e^{-\zeta\omega_nt}$。
* 臨界阻尼 $|\zeta|=1$：$y=(c_1+c_2t)e^{-\zeta\omega_nt}$，$c_1=y_0$、$c_2=v_0+\zeta\omega_ny_0$。
* 過阻尼 $|\zeta|>1$：$y=e^{-\zeta\omega_nt}\left(c_1e^{\zeta_1\omega_nt}+c_2e^{-\zeta_1\omega_nt}\right)$，$\zeta_1=\sqrt{\zeta^2-1}$。

（原檔第一行多了一個 `<html>` 標記，是網頁存檔時混進來的，MATLAB 執行前要刪掉；這裡已略去。）

In [2]:
def sys2ndfree(z, wn, y0, v0, tt):
    """逐行對應 sys2ndfree.m：回傳 (y, 包絡線或 None)"""
    if -1 < z < 1:
        wd = wn * np.sqrt(1 - z**2)
        c1 = y0
        c2 = (v0 + z * wn * y0) / wd
        env = np.sqrt(c1**2 + c2**2) * np.exp(-z * wn * tt)
        return np.exp(-z * wn * tt) * (c2 * np.sin(wd * tt) + c1 * np.cos(wd * tt)), env
    if abs(z) == 1:
        c1 = y0
        c2 = v0 + wn * z * y0
        return (c1 + c2 * tt) * np.exp(-wn * z * tt), None
    z1 = np.sqrt(z**2 - 1)
    c1 = (v0 + (z + z1) * wn * y0) / (2 * wn * z1)
    c2 = (-v0 - (z - z1) * wn * y0) / (2 * wn * z1)
    return np.exp(-z * wn * tt) * (c1 * np.exp(z1 * wn * tt) + c2 * np.exp(-z1 * wn * tt)), None


z, wn, y0, v0 = 0.3, 2.0, 1.0, 0.0
tt = np.arange(0, 10 + 1e-9, 0.01)                                  # 原程式 dt=0.001；畫圖用 0.01 已足夠
print("rt = roots([1 2*z*wn wn^2]) =", np.round(np.roots([1, 2 * z * wn, wn**2]), 4))
y, env = sys2ndfree(z, wn, y0, v0, tt)
fig = go.Figure()
fig.add_scatter(x=tt, y=y, name="y(t)", line=dict(color="#1f77b4", width=2.2))
fig.add_scatter(x=tt, y=env, name="envelope", line=dict(color="#d62728", dash="dot"))
fig.add_scatter(x=tt, y=-env, name="envelope", line=dict(color="#d62728", dash="dot"), showlegend=False)
show(fig, "sys2ndfree.m: zeta = 0.3, wn = 2, y(0) = 1, y'(0) = 0", ylab="y(t)")

rt = roots([1 2*z*wn wn^2]) = [-0.6+1.9079j -0.6-1.9079j]


改變阻尼比：拖動下方滑桿（$\omega_n=2$、$y_0=1$、$v_0=0$）。
每一種 $\zeta$ 都和數值積分比對，確認三種情形的公式（含係數 $c_1,c_2$）都正確。

In [3]:
zetas = [0.0, 0.1, 0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]
tt = np.linspace(0, 10, 1001)
fig = go.Figure()
for k, zz in enumerate(zetas):
    yk, _ = sys2ndfree(zz, wn, y0, v0, tt)
    yn = solve_ivp(lambda ti, s: [s[1], -2 * zz * wn * s[1] - wn**2 * s[0]], (0, 10), [y0, v0],
                   t_eval=tt, rtol=1e-11, atol=1e-12).y[0]
    print(f"zeta = {zz:3.1f}: max |formula - solve_ivp| = {np.max(np.abs(yk - yn)):.1e}")
    fig.add_scatter(x=tt, y=yk, name=f"zeta = {zz}", visible=(zz == 0.3), line=dict(width=2.2))
steps = [dict(method="update", label=f"{zz}", args=[{"visible": [i == k for i in range(len(zetas))]},
                                                     {"title": f"Free response, zeta = {zz}, wn = 2"}]) for k, zz in enumerate(zetas)]
fig.update_layout(sliders=[dict(active=zetas.index(0.3), currentvalue=dict(prefix="zeta = "), pad=dict(t=40), steps=steps)])
fig.update_yaxes(range=[-1.1, 1.1])
show(fig, "Free response, zeta = 0.3, wn = 2", ylab="y(t)", h=480)

zeta = 0.0: max |formula - solve_ivp| = 2.6e-11
zeta = 0.1: max |formula - solve_ivp| = 6.3e-12
zeta = 0.3: max |formula - solve_ivp| = 3.2e-12
zeta = 0.5: max |formula - solve_ivp| = 1.9e-12
zeta = 0.7: max |formula - solve_ivp| = 2.6e-12
zeta = 1.0: max |formula - solve_ivp| = 7.1e-13
zeta = 1.5: max |formula - solve_ivp| = 1.1e-12
zeta = 2.0: max |formula - solve_ivp| = 1.5e-12
zeta = 3.0: max |formula - solve_ivp| = 2.1e-12


## 2. 自由響應例題 `Osci1.m`、`nonOsc.m`

```matlab
% Osci1.m
t=0:0.05:10;
x=3*sin(2*t)+cos(2*t);
plot(t,x)
grid
```

```matlab
% nonOsc.m
t=0:0.05:5;
x=3*exp(-2*t)-2*exp(-3*t);
plot(t,x)
grid
```

例題 1：$\ddot x+4x=0,\ x(0)=1,\ \dot x(0)=6$，特徵根 $\pm j2$（無阻尼振盪）。
例題 2：$\ddot x+5\dot x+6x=0,\ x(0)=1,\ \dot x(0)=0$，特徵根 $-2,-3$（過阻尼、不振盪）。

In [4]:
x_osc1 = 3 * sp.sin(2 * t) + sp.cos(2 * t)
check("Osci1", 0, 4, 0, 1, 6, x_osc1, 10)
x_non = 3 * sp.exp(-2 * t) - 2 * sp.exp(-3 * t)
check("nonOsc", 5, 6, 0, 1, 0, x_non, 5)

fig = go.Figure()
tt = np.arange(0, 10 + 1e-9, 0.05)
fig.add_scatter(x=tt, y=num(x_osc1, tt), name="Osci1: x'' + 4x = 0")
tt5 = np.arange(0, 5 + 1e-9, 0.05)
fig.add_scatter(x=tt5, y=num(x_non, tt5), name="nonOsc: x'' + 5x' + 6x = 0")
show(fig, "Free response examples")

Osci1: sympy dsolve -> x(t) = 3*sin(2*t) + cos(2*t)
Osci1: agree with sympy = True, max |x - solve_ivp| = 7.8e-11
nonOsc: sympy dsolve -> x(t) = (3 - 2*exp(-t))*exp(-2*t)
nonOsc: agree with sympy = True, max |x - solve_ivp| = 6.6e-13


## 3. 強制響應：暫態解＋穩態解 `TranSteady.m`

```matlab
x = dsolve('D2x+5*Dx+6*x=10*sin(t)','x(0)=0,Dx(0)=0')
t=0:0.05:10;
xt=-exp(-3*t)+2*exp(-2*t);
xs=-cos(t)+sin(t);
x=-cos(t)+sin(t)-exp(-3*t)+2*exp(-2*t);
plot(t,xt,'b',t,xs,'r',t,x,'k')
grid
```

$\ddot x+5\dot x+6x=10\sin t,\ x(0)=0,\ \dot x(0)=0$。特徵根 $-2,-3$ 都在左半平面，
所以 $e^{-2t},e^{-3t}$ 項（暫態解，藍）會消失，最後只剩與輸入同頻率的 $-\cos t+\sin t$（穩態解，紅）。

sympy 的 `dsolve` 對應 MATLAB 的 `dsolve`；不給初始條件時會留下積分常數（網頁上的 Example 3）。

In [5]:
x_ts = -sp.cos(t) + sp.sin(t) - sp.exp(-3 * t) + 2 * sp.exp(-2 * t)
check("TranSteady", 5, 6, 10 * sp.sin(t), 0, 0, x_ts, 10)
print("Example 2, x(0)=1:  x(t) =", dsolve(5, 6, 10 * sp.sin(t), 1, 0))
print("Example 3, no I.C.: x(t) =", dsolve(5, 6, 10 * sp.sin(t)))
tt = np.arange(0, 10 + 1e-9, 0.05)
plot3(tt, [("xt: transient", num(-sp.exp(-3 * t) + 2 * sp.exp(-2 * t), tt)),
           ("xs: steady-state", num(-sp.cos(t) + sp.sin(t), tt)), ("x = xt + xs", num(x_ts, tt))],
      "TranSteady.m: x'' + 5x' + 6x = 10 sin t")

TranSteady: sympy dsolve -> x(t) = sin(t) - cos(t) + 2*exp(-2*t) - exp(-3*t)
TranSteady: agree with sympy = True, max |x - solve_ivp| = 4.0e-12
Example 2, x(0)=1:  x(t) = sin(t) - cos(t) + 5*exp(-2*t) - 3*exp(-3*t)
Example 3, no I.C.: x(t) = C1*exp(-3*t) + C2*exp(-2*t) + sin(t) - cos(t)


## 4. 不穩定系統 `Unstable.m`

```matlab
x=dsolve('D2x-5*Dx+6*x=10*sin(t)','x(0)=0,Dx(0)=0')
t=0:0.05:1;
xt=-2*exp(2*t)+exp(3*t);
xs=cos(t)+sin(t);
x=cos(t)+sin(t)-2*exp(2*t)+exp(3*t);
plot(t,xt,'b',t,xs,'r',t,x,'k')
grid
```

阻尼項變號：$\ddot x-5\dot x+6x=10\sin t$，特徵根 $+2,+3$ 在右半平面，$e^{2t},e^{3t}$ 隨時間發散，沒有穩態解可言。
原程式只畫到 $t=1$ 秒，畫久一點全解就被指數項淹沒。

In [6]:
x_un = sp.cos(t) + sp.sin(t) - 2 * sp.exp(2 * t) + sp.exp(3 * t)
check("Unstable", -5, 6, 10 * sp.sin(t), 0, 0, x_un, 1)
tt = np.arange(0, 1 + 1e-9, 0.05)
plot3(tt, [("xt: -2e^(2t) + e^(3t)", num(-2 * sp.exp(2 * t) + sp.exp(3 * t), tt)),
           ("xs: cos t + sin t", num(sp.cos(t) + sp.sin(t), tt)), ("x", num(x_un, tt))],
      "Unstable.m: x'' - 5x' + 6x = 10 sin t")

Unstable: sympy dsolve -> x(t) = exp(3*t) - 2*exp(2*t) + sin(t) + cos(t)
Unstable: agree with sympy = True, max |x - solve_ivp| = 3.9e-11


## 5. 無阻尼系統：共振與不共振 `Resonance.m`、`noResonance.m`、`Osci.m`

```matlab
% Resonance.m      x=simplify(dsolve('D2x+4*x=8*sin(2*t)','x(0)=0,Dx(0)=0'))
t=0:0.05:10;
xt=sin(2*t);
xs=-2*t.*cos(2*t);
x=-2*t.*cos(2*t)+sin(2*t);
plot(t,xt,'b',t,xs,'r',t,x,'k')
grid
```

```matlab
% noResonance.m    x=simplify(dsolve('D2x+4*x=3*sin(t)','x(0)=0,Dx(0)=0'))
t=0:0.05:10;
xt=-1/2*sin(2*t);
xs=sin(t);
x=sin(t)-1/2*sin(2*t);
plot(t,xt,'b',t,xs,'r',t,x,'k')
grid
```

```matlab
% Osci.m           x=simplify(dsolve('D2x+4*x=4','x(0)=0,Dx(0)=0'))
t=0:0.05:10;
x=1-cos(2*t);
plot(t,x)
grid
```

$\ddot x+4x$ 的自然頻率是 $\omega_n=2$ rad/s，而且沒有阻尼，所以齊次解 $\sin 2t,\cos 2t$ 永遠不會消失。
* 輸入頻率等於自然頻率（$8\sin 2t$）：特解是 $-2t\cos 2t$，振幅隨時間**線性增加**＝共振。
* 輸入頻率不等於自然頻率（$3\sin t$）：特解 $\sin t$ 的振幅固定，全解是兩個頻率的疊加。
* 步階輸入（$4$）：$x=1-\cos 2t$，在 0 與 2 之間不停振盪，平均值是靜態位移 1。

In [7]:
x_res = -2 * t * sp.cos(2 * t) + sp.sin(2 * t)
check("Resonance", 0, 4, 8 * sp.sin(2 * t), 0, 0, x_res, 10)
x_nores = sp.sin(t) - sp.Rational(1, 2) * sp.sin(2 * t)
check("noResonance", 0, 4, 3 * sp.sin(t), 0, 0, x_nores, 10)
x_osci = 1 - sp.cos(2 * t)
check("Osci", 0, 4, 4, 0, 0, x_osci, 10)

tt = np.arange(0, 10 + 1e-9, 0.05)
plot3(tt, [("xt: sin 2t", num(sp.sin(2 * t), tt)), ("xs: -2t cos 2t", num(-2 * t * sp.cos(2 * t), tt)), ("x", num(x_res, tt))],
      "Resonance.m: x'' + 4x = 8 sin 2t")
plot3(tt, [("xt: -1/2 sin 2t", num(-sp.sin(2 * t) / 2, tt)), ("xs: sin t", num(sp.sin(t), tt)), ("x", num(x_nores, tt))],
      "noResonance.m: x'' + 4x = 3 sin t")
fig = go.Figure()
fig.add_scatter(x=tt, y=num(x_osci, tt), name="x = 1 - cos 2t")
show(fig, "Osci.m: x'' + 4x = 4")

Resonance: sympy dsolve -> x(t) = -2*t*cos(2*t) + sin(2*t)
Resonance: agree with sympy = True, max |x - solve_ivp| = 2.2e-10
noResonance: sympy dsolve -> x(t) = sin(t) - sin(2*t)/2
noResonance: agree with sympy = True, max |x - solve_ivp| = 2.4e-11


Osci: sympy dsolve -> x(t) = 1 - cos(2*t)
Osci: agree with sympy = True, max |x - solve_ivp| = 3.8e-11
